# PoreMind: 逐行逐步分析示例
该 Notebook 演示：读取 ABF/CSV -> 预处理 -> 事件表 -> 标注训练 -> 新样本逐事件判定。

In [ ]:
from poremind.pipeline import AnalysisConfig, analyze_abf_to_event_df
from poremind.ml import LabeledDataset, train_event_classifier, predict_events

In [ ]:
config = AnalysisConfig(
    reader='abf',
    preprocess_method='drift_corrected_moving_average',
    preprocess_params={'drift_window': 1001, 'smooth_window': 5},
    baseline_method='rolling_quantile',
    baseline_params={'window': 501, 'q': 0.5},
    detect_params={'sigma_k': 5.0, 'min_duration_s': 2e-4},
)

In [ ]:
events_df = analyze_abf_to_event_df('your_file.abf', config=config, channel=0, sweep=0)
events_df.head()

## 标注样本并训练模型
在 events_df 里新增 `label` 列（例如标准品类别），然后训练分类器。

In [ ]:
# 示例：手工或程序化写入 label
# events_df.loc[events_df['event_id'].isin([1,2,3]), 'label'] = 'standard_A'
# events_df.loc[events_df['event_id'].isin([4,5]), 'label'] = 'standard_B'

dataset = LabeledDataset(events_df.dropna(subset=['label']))
model_pkg = train_event_classifier(dataset, model_name='random_forest', model_params={'n_estimators': 300, 'random_state': 42})

In [ ]:
new_df = analyze_abf_to_event_df('new_sample.abf', config=config, channel=0, sweep=0)
pred_df = predict_events(model_pkg, new_df)
pred_df[['event_id', 'start_time_s', 'end_time_s', 'pred_label']].head()